In [ ]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from tqdm.notebook import tqdm_notebook
import gc

from huggingface_hub import HfApi, hf_hub_download

from mech_interp_toolkit.activation_utils import concat_activations

In [ ]:
repo_id = "AISC-Linear-Probe-Gen/obfuscated_activations"
max_files = 1e6

api = HfApi()

# Auto-discover all folders in the repo
top_level = list(api.list_repo_tree(repo_id=repo_id, repo_type="dataset"))
folders = sorted([item.rfilename for item in top_level if item.type == "directory"])
print(f"Found folders: {folders}")

# Download activations for each folder
all_activations = {}  # folder_name -> list of activation dicts

for folder in folders:
    print(f"\nDownloading from '{folder}'...")
    folder_files = list(api.list_repo_tree(repo_id=repo_id, repo_type="dataset", path_in_repo=folder))
    pt_files = sorted([f.rfilename for f in folder_files if f.rfilename.endswith(".pt")])

    activations = []
    for i, filepath in enumerate(tqdm_notebook(pt_files, desc=folder)):
        if i >= max_files:
            break
        local_path = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=filepath)
        act = torch.load(local_path, weights_only=False)
        act.attention_mask = torch.empty(0)
        activations.append(act)

    all_activations[folder] = activations
    print(f"  Loaded {len(activations)} files")

In [ ]:
# Pre-compute activations for all layers, for each folder
all_layers = {}  # folder_name -> {layer: numpy array}

for folder, acts_list in all_activations.items():
    print(f"Processing '{folder}'...")
    merged = concat_activations(acts_list, pad_value=0)
    layers_dict = {}
    for layer in tqdm_notebook(range(28), desc=folder):
        layers_dict[layer] = merged[(layer, "layer_out")][:, -1, :].float().numpy()
    all_layers[folder] = layers_dict
    del merged

del all_activations
gc.collect()
torch.cuda.empty_cache()

print(f"\nReady — activation types: {list(all_layers.keys())}")

In [ ]:
COLORS = plt.cm.tab10.colors

def plot_pca_for_layer(layer):
    """Plot PCA cumulative variance for all activation types at a given layer."""
    folder_names = list(all_layers.keys())

    plt.figure(figsize=(12, 6))

    # Reference lines
    plt.axhline(y=0.90, color='red', linestyle='--', linewidth=1.5, label='90% variance', alpha=0.5)
    plt.axhline(y=0.95, color='green', linestyle='--', linewidth=1.5, label='95% variance', alpha=0.5)

    max_components = 0

    for idx, folder in enumerate(folder_names):
        color = COLORS[idx % len(COLORS)]
        data = all_layers[folder][layer]

        pca = PCA()
        pca.fit(data)
        cumvar = np.cumsum(pca.explained_variance_ratio_)
        max_components = max(max_components, len(cumvar))

        n90 = int(np.argmax(cumvar >= 0.90)) + 1
        n95 = int(np.argmax(cumvar >= 0.95)) + 1

        plt.plot(range(1, len(cumvar) + 1), cumvar, linewidth=2, label=folder, color=color)
        plt.axvline(x=n90, color=color, linestyle=':', linewidth=1, alpha=0.4)
        plt.axvline(x=n95, color=color, linestyle=':', linewidth=1, alpha=0.4)

        # Stagger annotation vertically per folder to reduce overlap
        y_offset_90 = 0.87 - 0.03 * idx
        y_offset_95 = 0.96 - 0.03 * idx
        plt.text(n90, y_offset_90, f'{folder}: {n90}', ha='center', va='top', fontsize=8, color=color,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=color, alpha=0.7))
        plt.text(n95, y_offset_95, f'{folder}: {n95}', ha='center', va='top', fontsize=8, color=color,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor=color, alpha=0.7))

    plt.xlabel('Number of Components', fontsize=12)
    plt.ylabel('Cumulative Variance Explained', fontsize=12)
    plt.title(f'PCA Cumulative Variance — All Activation Types (Layer {layer})', fontsize=14)
    plt.legend(loc='lower right', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, max_components)
    plt.ylim(0, 1.0)
    plt.tight_layout()
    plt.show()

    for folder in folder_names:
        data = all_layers[folder][layer]
        pca = PCA()
        pca.fit(data)
        cumvar = np.cumsum(pca.explained_variance_ratio_)
        n90 = int(np.argmax(cumvar >= 0.90)) + 1
        n95 = int(np.argmax(cumvar >= 0.95)) + 1
        print(f"{folder}:  90% → {n90} components,  95% → {n95} components,  total → {len(cumvar)}")

widgets.interact(plot_pca_for_layer, layer=widgets.IntSlider(min=0, max=27, step=1, value=17, description='Layer:'));